# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DEEPIKA-Inag/Flyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Scoring.** The lane is Refresh / Content Opportunity Scoring: with ~30k live pages, editors need a continuous risk-of-decline score to sort into a ranked review queue, not just a yes/no label. Review capacity is fixed each week, so what matters is who sits near the top of the list -- that is a scoring problem. Ranking doesn't fit (there's no query to rank results for); classification is a step inside scoring (it supplies the probability), not the end product.

In [ ]:
import os, sys, subprocess
import pandas as pd, numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DEEPIKA-Inag/Flyrank-internship"
REPO_DIR = "Flyrank-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{df.shape[0]:,} pages x {df.shape[1]} columns -- too many to eyeball one by one each week, which is exactly why this needs a score to sort them.")

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy, built on an observed outcome.** Target = `is_declining_label = (trend_direction == "down")`. `trend_direction` is computed by comparing two observed GSC windows -- `impressions_last_30d` vs `impressions_prev_30d` -- so the underlying signal is real traffic, not a human guess. But "down" is a *threshold* cut on that comparison (54.2% of rows land there, per the cell below), so the label is a defined rule layered on top of an observed number -- a threshold I inherited from the pipeline, not one I've verified myself yet.

In [ ]:
is_declining = (df["trend_direction"] == "down")
print(df["trend_direction"].value_counts())
print(f"\nShare labeled 'down': {is_declining.mean():.1%}")

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50.** Of the top 50 pages my score sends to an editor this week, what share are actually declining? It's defensible because it matches the real constraint (fixed weekly review capacity, not a global classification target), and it's computable today from this CSV alone -- no live experiment needed. The repo's own reference pipeline (`outputs/model_report.md`) already reports exactly this metric on this dataset: 0.240 for a hand-written baseline rule vs 0.740 for a trained random forest. That gives me a number to both defend and beat.

In [ ]:
# A naive single-signal rule: "review the stalest pages first"
naive_rank = df["days_since_last_update"].fillna(0).sort_values(ascending=False)
top50 = naive_rank.index[:50]
precision_at_50_naive = is_declining.loc[top50].mean()
print(f"Naive 'staleness-only' rule, Precision@50: {precision_at_50_naive:.3f}")
print(f"Overall decline rate (random-order baseline): {is_declining.mean():.3f}")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one published content page** (`content_id`), aggregated over a trailing 90-day window, tied to a single client (`client_id`). Loaded and confirmed below.

In [ ]:
print(df.shape)
assert df["content_id"].is_unique, "expected one row per content page"
df[["content_id", "client_id", "content_type", "main_intent",
    "position_tier", "freshness_tier", "trend_direction"]].head(5)

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Because the useful signal is an interaction, not one cutoff.** Below, pages sitting in the top 3 results swing from a 17.5% decline rate when fresh (updated in the last 30 days) to 67.9% when stale (91-180 days) -- freshness matters enormously there. But for pages buried on page 3+ ('deep'), freshness barely moves the needle: 34.0% -> 35.9%. A single if-statement ('flag anything not updated in 90 days') would either over-flag deep pages or under-flag top-ranked ones -- it can't hold two different thresholds for two different segments at once, and that's before adding content_type, intent, or engagement into the mix.

In [ ]:
me content-refresh scoring task

In [ ]:
interaction = (
    df.assign(is_declining=is_declining)
      .groupby(["position_tier", "freshness_tier"])["is_declining"]
      .agg(decline_rate="mean", n="size")
      .round(3)
)
print(interaction.loc[["top_3", "deep"]])

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.